# 03 - Label Transfer of Leiden Clusters from Big Screen

## Imports and configuration

In [ ]:
import os
import warnings

import numpy as np
import pandas as pd
import scanpy as sc

from config import (
    ANNDATA_BIG_SCREEN,
    ANNDATA_LEIDEN_TC_TRANSFERRED,
    ANNDATA_LEIDEN_TC_TRANSFERRED_REGOUT,
    ANNDATA_PROCESSED,
    LEIDEN_TOP_MARKERS_CSV,
    NEIGHBOR_METRIC,
    N_NEIGHBORS,
    N_PCS,
    N_REGRESS_JOBS,
    PROJECT_DIR,
)

warnings.filterwarnings("ignore")
sc.settings.verbosity = 2
sc.settings.set_figure_params(dpi=100, frameon=False)

os.chdir(PROJECT_DIR)

## 1. Load data

In [ ]:
ANNDATA_PROCESSED

In [ ]:
adata = sc.read(ANNDATA_PROCESSED)
adata

In [ ]:
ANNDATA_BIG_SCREEN

In [ ]:
adataBigScreen = sc.read(ANNDATA_BIG_SCREEN)
adataBigScreen.obs["leidenBigScreen"] = adataBigScreen.obs["leiden"]
adataBigScreen.obs.shape

## 2. Restrict to common genes

In [ ]:
commonGenes = adataBigScreen.var_names[[x in adata.var_names for x in adataBigScreen.var_names]]
len(commonGenes)

In [ ]:
adataBigScreen = adataBigScreen[:, commonGenes]
adata = adata[:, commonGenes]

## 3. Transfer Leiden labels via `sc.tl.ingest`

In [ ]:
sc.pp.neighbors(adata, n_pcs=50)

In [ ]:
sc.pp.neighbors(adataBigScreen, n_pcs=50)

In [ ]:
sc.tl.ingest(adata, adataBigScreen, obs="leidenBigScreen")

In [ ]:
sc.pl.umap(
    adata,
    color="leidenBigScreen",
    color_map="coolwarm",
    legend_fontoutline=3,
    legend_fontsize=14,
    legend_fontweight="normal",
    size=3,
)

## 4. Attach transferred labels to processed adata

In [ ]:
leidenBigScreen = pd.DataFrame(adata.obs["leidenBigScreen"])
leidenBigScreen

In [ ]:
adata = sc.read(ANNDATA_PROCESSED)
adata.obs["leidenBigScreen"] = leidenBigScreen.leidenBigScreen

In [ ]:
sc.pl.umap(
    adata,
    color="leidenBigScreen",
    color_map="coolwarm",
    legend_fontoutline=3,
    legend_loc="on data",
    legend_fontsize=14,
    legend_fontweight="normal",
    size=3,
)

In [ ]:
adata.write(ANNDATA_LEIDEN_TC_TRANSFERRED)